# GAN — Generative Adversarial Network\nDCGAN ile MNIST rakam üretimi. Generator sahte rakam üretir, Discriminator gerçek/sahte ayırt eder.\nBirbirleriyle yarışarak gelişirler.

In [ ]:
import torch, torch.nn as nn, torch.optim as optim\nimport torchvision, torchvision.transforms as T\nimport matplotlib.pyplot as plt, numpy as np, os\nos.makedirs('cikti', exist_ok=True)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\nprint(f'Cihaz: {device}')

## MNIST Veri Seti ve DataLoader

In [ ]:
transform = T.Compose([T.ToTensor(), T.Normalize((0.5,), (0.5,))])\ndataset = torchvision.datasets.MNIST(root='./data', train=True, download=True, transform=transform)\nloader = torch.utils.data.DataLoader(dataset, batch_size=64, shuffle=True)\nprint(f'Veri: {len(dataset)} görüntü')

## Generator ve Discriminator

In [ ]:
LATENT = 100\n\nclass Generator(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(LATENT, 256), nn.ReLU(),\n            nn.Linear(256, 512), nn.ReLU(),\n            nn.Linear(512, 1024), nn.ReLU(),\n            nn.Linear(1024, 784), nn.Tanh(),\n        )\n    def forward(self, z):\n        return self.net(z).view(-1, 1, 28, 28)\n\nclass Discriminator(nn.Module):\n    def __init__(self):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Flatten(),\n            nn.Linear(784, 512), nn.LeakyReLU(0.2),\n            nn.Linear(512, 256), nn.LeakyReLU(0.2),\n            nn.Linear(256, 1), nn.Sigmoid(),\n        )\n    def forward(self, x):\n        return self.net(x)\n\nG = Generator().to(device)\nD = Discriminator().to(device)\nprint(f'G: {sum(p.numel() for p in G.parameters()):,} param | D: {sum(p.numel() for p in D.parameters()):,} param')

## Eğitim

In [ ]:
criterion = nn.BCELoss()\ng_opt = optim.Adam(G.parameters(), lr=0.0002, betas=(0.5, 0.999))\nd_opt = optim.Adam(D.parameters(), lr=0.0002, betas=(0.5, 0.999))\n\ng_losses, d_losses = [], []\nEPOCHS = 20\nfixed_z = torch.randn(16, LATENT, device=device)\n\nfor epoch in range(EPOCHS):\n    g_loss_epoch, d_loss_epoch = 0, 0\n    for real_imgs, _ in loader:\n        batch = real_imgs.size(0)\n        real_imgs = real_imgs.to(device)\n        real_labels = torch.ones(batch, 1, device=device) * 0.9  # label smoothing\n        fake_labels = torch.zeros(batch, 1, device=device)\n\n        # Discriminator eğitimi\n        D.zero_grad()\n        d_real = criterion(D(real_imgs), real_labels)\n        z = torch.randn(batch, LATENT, device=device)\n        fake = G(z).detach()\n        d_fake = criterion(D(fake), fake_labels)\n        d_loss = d_real + d_fake\n        d_loss.backward()\n        d_opt.step()\n\n        # Generator eğitimi\n        G.zero_grad()\n        z = torch.randn(batch, LATENT, device=device)\n        fake = G(z)\n        g_loss = criterion(D(fake), real_labels)\n        g_loss.backward()\n        g_opt.step()\n\n        g_loss_epoch += g_loss.item()\n        d_loss_epoch += d_loss.item()\n    \n    g_losses.append(g_loss_epoch / len(loader))\n    d_losses.append(d_loss_epoch / len(loader))\n    print(f'Epoch {epoch+1:2d}/{EPOCHS} | G Loss: {g_losses[-1]:.4f} | D Loss: {d_losses[-1]:.4f}')

## Kayıp Grafiği

In [ ]:
plt.figure(figsize=(10, 4))\nplt.plot(g_losses, label='Generator'); plt.plot(d_losses, label='Discriminator')\nplt.xlabel('Epoch'); plt.ylabel('Loss'); plt.legend(); plt.title('GAN Eğitim Kaybı')\nplt.savefig('cikti/gan_loss.png', dpi=100, bbox_inches='tight')\nplt.show()

## Üretilen Rakamlar

In [ ]:
G.eval()\nwith torch.no_grad():\n    generated = G(fixed_z).cpu()\n\nfig, axes = plt.subplots(2, 8, figsize=(12, 3))\nfor i, ax in enumerate(axes.flat):\n    ax.imshow(generated[i].squeeze(), cmap='gray')\n    ax.axis('off')\nplt.suptitle('GAN ile Üretilen MNIST Rakamları (20 epoch)')\nplt.savefig('cikti/gan_uretilen.png', dpi=100, bbox_inches='tight')\nplt.show()\nprint('GAN eğitimi tamamlandı.')